In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!rm -rf sophia-romberg-data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, accuracy_score
from numpy import mean
from numpy import std
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

In [ ]:
!git clone https://github.com/A2AppRom/sophia-romberg-data

Cloning into 'sophia-romberg-data'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 105 (delta 20), reused 73 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 4.81 MiB | 17.65 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [ ]:
# define paths to train, test split
DATA_ROOT = "/content/sophia-romberg-data/labeled_data_with_features_FINAL_FINAL.csv"

In [ ]:
df = pd.read_csv(DATA_ROOT)

In [ ]:
df.columns

Index(['Unnamed: 0', 'subject_id', 'session_id', 'label', 'duration', 'mean',
       'median', 'std', 'skew', 'kurtosis'],
      dtype='object')

In [ ]:
df = df.drop(columns="Unnamed: 0")

In [ ]:
# determine need for class weighting
targets = ["label"]

for t in targets:
    print(t)
    print(df[t].value_counts(normalize=True))
    print()

label
label
open      0.5
closed    0.5
Name: proportion, dtype: float64



In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

print("Numeric:", len(numeric_cols))
print("Categorical:", len(categorical_cols))

Numeric: 8
Categorical: 1


In [ ]:
# convert label to numerical
labels = df['label'].copy()

for i in range(len(labels)):
  if labels[i] == 'open':
    labels[i] = 0
  else:
    labels[i] = 1

In [ ]:
df_reg = df.copy()
df_reg['label'] = labels

In [ ]:
df_reg['label'].value_counts()

,count
label,
0,67
1,67


In [ ]:
df_reg.head()

,subject_id,session_id,label,duration,mean,median,std,skew,kurtosis
0,1,0,0,58251602400,0.121713,0.060539,0.334678,9.272823,133.034741
1,1,0,1,45065531600,0.183694,0.104852,0.318721,5.781224,42.265251
2,2,0,0,42344970200,0.134327,0.073325,0.247356,5.967011,43.137762
3,2,0,1,42567076100,0.128552,0.082975,0.226567,7.675751,75.174710
4,3,0,0,42296806700,0.118685,0.071725,0.216500,7.366179,71.688904


normalize data

In [ ]:
cols_to_norm = ['duration']
df[cols_to_norm] = df[cols_to_norm].apply(lambda x: (x - x.min()) / (x.max() - x.min()))

In [ ]:
# if using continuous labels
cols_to_norm = ['duration']
df_reg[cols_to_norm] = df_reg[cols_to_norm].apply(lambda x: (x - x.min()) / (x.max() - x.min()))

In [ ]:
df_reg.head()

,subject_id,session_id,label,duration,mean,median,std,skew,kurtosis
0,1,0,0,1.000000,0.121713,0.060539,0.334678,9.272823,133.034741
1,1,0,1,0.608323,0.183694,0.104852,0.318721,5.781224,42.265251
2,2,0,0,0.527511,0.134327,0.073325,0.247356,5.967011,43.137762
3,2,0,1,0.534109,0.128552,0.082975,0.226567,7.675751,75.174710
4,3,0,0,0.526081,0.118685,0.071725,0.216500,7.366179,71.688904


In [ ]:
results = {}

gridsearch for KNN to find best params

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
targets = ["label"]
X = df.drop(columns=targets)
y = df["label"]  # start with one horizon

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
# stratify maintains class proportions and random_state controls seed of PRG (pseudorandom generator)

knn = KNeighborsClassifier()

param_grid = {
    'n_neighbors': [i for i in range (3, 2000) if i % 2 != 0],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

model = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')

model.fit(X_train, y_train)

print("best params:", model.best_params_)

best params: {'metric': 'manhattan', 'n_neighbors': 21, 'weights': 'distance'}


In [ ]:
results.update({"KNN best params": str(model.best_params_)})

KNN classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
targets = ["label"]
X = df.drop(columns=targets)
y = df["label"]  # start with one horizon

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
# stratify maintains class proportions and random_state controls seed of PRG (pseudorandom generator)

model = KNeighborsClassifier(metric='manhattan', n_neighbors=21, weights='distance')
model.fit(X_train, y_train)

val_probs = model.predict_proba(X_val)[:, 1]

roc_auc = roc_auc_score(y_val, val_probs)
accuracy = accuracy_score(y_val, model.predict(X_val))
print("ROC AUC:", roc_auc)
print("Accuracy score:", accuracy)
results.update({"KNN Classifier Accuracy": str(accuracy)})

ROC AUC: 0.7304219157572142
Accuracy score: 0.6457073760580411


SVM Classification + GroupKfold + GridSearchCV

In [ ]:
from sklearn import svm
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
targets = ["label"]
X = df_reg.drop(columns=targets)
y = df_reg["label"].astype(int)  # Ensure y is of integer type

X_scaled = StandardScaler().fit_transform(X)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_encoded, test_size=0.2, stratify=y, random_state=42)

In [ ]:
# scale data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [ ]:
model = svm.SVC(kernel="linear", probability=True, class_weight="balanced")
model.fit(X_train, y_train)

val_probs = model.predict_proba(X_val)[:, 1]
accuracy = accuracy_score(y_val, model.predict(X_val))
#print(classification_report(y_val, model.predict(X_val)))
print("Accuracy score:", accuracy)

Accuracy score: 0.5925925925925926


In [ ]:
results.update({"SVM Linear Accuracy:": accuracy})

In [ ]:
model = svm.SVC()
gkf = GroupKFold(n_splits=2)
groups = df_reg["subject_id"]

param_grid = {
    "C": [0.1, 1.0, 10],
    "kernel": ["linear", "rbf"]
}

In [ ]:
grid = GridSearchCV(
    svm.SVC(),
    param_grid,
    cv=gkf,
    n_jobs=-1
)

In [ ]:
grid.fit(X, y, groups=groups)

KeyboardInterrupt: 

In [ ]:
print(grid.best_params_)

In [ ]:
grid_predictions = grid.predict(X_val)

print(classification_report(y_val, grid_predictions))

linear classifier

In [ ]:
from sklearn import linear_model

In [ ]:
targets = ["label"]
X = df_reg.drop(columns=targets)
y = df_reg["label"].astype(int)  # Ensure y is of integer type

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

model = linear_model.LinearRegression()
model.fit(X_train, y_train)

val_probs = model.predict(X_val)

# Convert continuous predictions to binary labels for accuracy score
predicted_classes = (val_probs > 0.5).astype(int)

accuracy = accuracy_score(y_val, predicted_classes)
print("ROC AUC:", roc_auc_score(y_val, val_probs))
print("Accuracy:", accuracy)
results.update({"Linear Classifier Accuracy:": str(accuracy)})

ROC AUC: 0.6381724392041268
Accuracy: 0.6203143893591294


random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
targets = ["label"]
X = df_reg.drop(columns=targets)
y = df_reg["label"].astype(int)  # Ensure y is of integer type

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

model = RandomForestClassifier(n_estimators=10000, max_depth=1000,
                               random_state=0, bootstrap=True)
model.fit(X_train, y_train)

val_probs = model.predict_proba(X_val)[:, 1]

accuracy = accuracy_score(y_val, model.predict(X_val))
print("ROC AUC:", roc_auc_score(y_val, val_probs))
print("Accuracy:", accuracy)
results.update({"Random Forest Accuracy:": str(accuracy)})

ROC AUC: 0.8779345194230972
Accuracy: 0.7787182587666264


LogisticRegression

In [ ]:
targets = ["label"]
X = df.drop(columns=targets)
y = df["label"]  # start with one horizon

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(exclude=np.number).columns

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:
model = Pipeline([
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

model.fit(X_train, y_train)

val_probs = model.predict_proba(X_val)[:, 1]

In [ ]:
accuracy = accuracy_score(y_val, model.predict(X_val))
print("Accuracy:", accuracy)
print("ROC AUC:", roc_auc_score(y_val, val_probs))
results.update({"Logistic Regression Accuracy:": str(accuracy)})

Accuracy: 0.6203143893591294
ROC AUC: 0.6498169397948323


In [ ]:
y_val

,label
29,closed
21,closed
117,closed
124,open
28,open
118,open
47,closed
14,open
105,closed
77,closed


In [ ]:
val_probs

array([0.50159286, 0.5246394 , 0.51857644, 0.48852787, 0.50543199,
       0.52412312, 0.69385644, 0.46470852, 0.55357057, 0.50477177,
       0.4723265 , 0.51989205, 0.65686012, 0.45889775, 0.49672498,
       0.48343466, 0.50488998, 0.53242184, 0.44430628, 0.4836249 ,
       0.43473398, 0.56402887, 0.50272485, 0.47960691, 0.52272844,
       0.44954981, 0.40027035])

HistGradientBoostingClassifier

In [ ]:
model = HistGradientBoostingClassifier(max_bins=255, max_iter=100)
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
# evaluate the model and collect the scores
n_scores = cross_val_score(model, X, y, scoring='accuracy', cv=cv, n_jobs=-1)
print('Accuracy: %.3f (%.3f)' % (mean(n_scores), std(n_scores)))
results.update({"HistGradientBoostingClassifier w/ Cross Val Accuracy:": str(mean(n_scores))})

Accuracy: 0.755 (0.018)


In [ ]:
results

{'KNN best params': "{'metric': 'manhattan', 'n_neighbors': 21, 'weights': 'distance'}",
 'KNN Classifier Accuracy': '0.6457073760580411',
 'SVM Linear Accuracy:': 0.619105199516324,
 'Linear Classifier Accuracy:': '0.6203143893591294',
 'Random Forest Accuracy:': '0.7787182587666264',
 'Logistic Regression Accuracy:': '0.6203143893591294',
 'HistGradientBoostingClassifier w/ Cross Val Accuracy:': '0.7548687386196598'}

XGBoost

In [ ]:
!sudo pip install xgboost

training

In [ ]:
targets = ["label"]
X = df_reg.drop(columns=targets)
y = df_reg["label"].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = XGBClassifier(tree_method='approx', max_bin=255, n_estimators=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)
print("ROC AUC:", roc_auc_score(y_val, y_pred))

ROC AUC: 0.5576923076923077


cross validation

In [ ]:
# define the evaluation procedure
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)

# evaluate the model and collect the scores
n_scores = cross_val_score(model, X, y, scoring='accuracy', cv=cv, n_jobs=-1)
# report performance
print('Accuracy: %.3f (%.3f)' % (mean(n_scores), std(n_scores)))

Accuracy: 0.610 (0.131)
